In [1]:
from reflex_database import LoadChunkFileForUsers,LoadFileForUsers, LoadEyeGazeForUsers,is_csv_file,getArrFromCsv,getStrArrFromCsv
import pandas as pd
import os
import json
from clustermain import Cluster


maxUser = 16
path = 'C:/Users/z5308/Desktop/VRTestingProject/data/Reflex_data/simple_3joints/'
player_dataset = LoadFileForUsers(path)
path2 = 'C:/Users/z5308/Desktop/VRTestingProject/data/Reflex_data/simple_3joints/otherInfo'
chunk_database = LoadChunkFileForUsers(path2)





c:\Users\z5308\.conda\envs\vrtest\lib\site-packages\tslearn\bases\bases.py:15: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


Openning player0.csv
Openning player1.csv
Openning player2.csv
Openning player3.csv
Openning player4.csv
Openning player5.csv
Openning player6.csv
Openning player7.csv
Openning player8.csv
Openning player9.csv
Openning player10.csv
Openning player11.csv
Openning player12.csv
Openning player13.csv
Openning player14.csv
Openning player15.csv
Openning player0_chunk.csv
Openning player1_chunk.csv
Openning player2_chunk.csv
Openning player3_chunk.csv
Openning player4_chunk.csv
Openning player5_chunk.csv
Openning player6_chunk.csv
Openning player7_chunk.csv
Openning player8_chunk.csv
Openning player9_chunk.csv
Openning player10_chunk.csv
Openning player11_chunk.csv
Openning player12_chunk.csv
Openning player13_chunk.csv
Openning player14_chunk.csv
Openning player15_chunk.csv


In [4]:
def ObtainSelectRangeDiscreteData(startChunk, endChunk, maxUser, chunk_database):
    filter_database = chunk_database[(chunk_database['chunkType'] >= startChunk ) & (chunk_database['chunkType'] <= endChunk)]
    
    df = pd.DataFrame(columns = ['userID','GameplayDur' ,'NumCoinsCollected', 'NumBombHit', 
                        'NumObstaclesHit','NumTreasureCollected',
                         'norNumCoinsCollected','norNumBombHit', 'norNumTreasureCollected',])
    for userID in range(maxUser):

        numCoinsCollected = filter_database[(filter_database['userID'] == userID) ]['numCoinsCollected'].sum()
        numBombHit = filter_database[(filter_database['userID'] == userID) ]['numBombHit'].sum()
        numObstaclesHit= filter_database[(filter_database['userID'] == userID) ]['numObstaclesHit'].sum()
        numTreasureCollected = filter_database[(filter_database['userID'] == userID) ]['numTreasureCollected'].sum()

        numCoinsHave = filter_database[(filter_database['userID'] == userID) ]['numCoinsHave'].sum()
        numBomb = filter_database[(filter_database['userID'] == userID) ]['numBomb'].sum()
        numTreasureHave = filter_database[(filter_database['userID'] == userID) ]['numTreasureHave'].sum()
        
        normNumCoinsCollected = numCoinsCollected/ numCoinsHave if numCoinsHave !=0 else 0
        norNumBombHit =  numBombHit/ numBomb if numBomb !=0 else 0
        norNumTreasureCollected = numTreasureCollected/ numTreasureHave if numTreasureHave !=0 else 0

        GameplayDur = filter_database[(filter_database['userID'] == userID) & (filter_database['chunkType'] == endChunk)]['endFrame'].astype(int).iloc[0]/30.0
       
        data = {
            "userID": [userID],
            'GameplayDur':[GameplayDur],
            "NumCoinsCollected": [numCoinsCollected],
            "NumBombHit": [numBombHit],
            "NumObstaclesHit": [numObstaclesHit],
            "NumTreasureCollected": [numTreasureCollected],

            'norNumCoinsCollected':[normNumCoinsCollected],
            'norNumBombHit':[norNumBombHit],
            'norNumTreasureCollected':[norNumTreasureCollected],
     
        }
        # Create a DataFrame from the data dictionary
        new_df = pd.DataFrame(data)
        
        # Concatenate the new data to the existing DataFrame
        df = pd.concat([df, new_df], ignore_index=True)
    return df


clusterSetting = None


startChunk = int(clusterSetting['startChunk']) 
endChunk = int(clusterSetting['endChunk']) 

discrete_df = ObtainSelectRangeDiscreteData(startChunk, endChunk, maxUser,chunk_database)
discrete_df[['userID','NumObstaclesHit']]




,userID,NumObstaclesHit
0,0,0
1,1,0
2,2,0
3,3,10
4,4,7
5,5,3
6,6,0
7,7,0
8,8,0
9,9,0


In [8]:
with open("./Setting/ClusterSetting.json") as json_file:
			clusterSetting = json.load(json_file)

In [9]:
import numpy as np
from sklearn.preprocessing import normalize, StandardScaler
from clustermain import Cluster,ConstructTrainingSet2,AppendFeature,GetOtherFeatureArr,RunKMeanCluster,RunElasticNetSSC,RunHDBSCANCluster





features = clusterSetting['Features'] #get feature
numcluster = int(clusterSetting['NumCluster']) 
min_PlayerPerCluster = 3#int(clusterSetting['Min_PlayerPerCluster'])

sdf = chunk_database[chunk_database['chunkType'] == startChunk]
startFrames = sdf[['userID', 'startFrame']]
sdf = chunk_database[chunk_database['chunkType'] == endChunk]
endFrames = sdf[['userID','endFrame']]

X_train = None
if(features['WalkPath']):
    X_train, C = ConstructTrainingSet2(player_dataset, startFrames, endFrames)

features = clusterSetting['Features'] #get feature
extraFeature = GetOtherFeatureArr(features)
X_train = AppendFeature(discrete_df, extraFeature, X_train)

print(X_train)
y_predit = None
if(clusterSetting['Pose_Algo'] == "ElasticNet"):
    y_predit = RunElasticNetSSC(X_train, numcluster)
    print("Run ElasticNet")
elif(clusterSetting['Pose_Algo'] =="KMean" ):
    y_predit,_ = RunKMeanCluster(X_train, numcluster)
    print("Run Kmean")
else: #
    y_predit = RunHDBSCANCluster(X_train, 3, min_PlayerPerCluster)
    print("Run HDBSCAN")

discrete_df['GroupID'] = y_predit
columns = ['userID','GroupID'] + extraFeature
final_df = discrete_df[columns]
#final_df.to_csv('C:/Users/z5308/Desktop/VRTestingProject/data/Reflex_data/simple_3joints/FinalInfo.csv', index=False)
final_df

feature to do: [NumObstaclesHit,]
[[-0.52802501]
 [-0.52802501]
 [-0.52802501]
 [ 2.28810837]
 [ 1.44326835]
 [ 0.316815  ]
 [-0.52802501]
 [-0.52802501]
 [-0.52802501]
 [-0.52802501]
 [-0.52802501]
 [-0.52802501]
 [-0.52802501]
 [ 2.28810837]
 [-0.52802501]
 [-0.52802501]]
Run Kmean


,userID,GroupID,NumObstaclesHit
0,0,1,0
1,1,1,0
2,2,1,0
3,3,0,10
4,4,0,7
5,5,1,3
6,6,1,0
7,7,1,0
8,8,1,0
9,9,1,0


In [ ]:

columns

In [ ]:
from SSC_algo.cluster.selfrepresentation import ElasticNetSubspaceClustering
from sklearn.preprocessing import StandardScaler
from clustermain import RunKMeanCluster
X = [[0.25 ], [0.   ],[0.875],[0.625],[0.875],
 [0.   ],[0.   ],[0.625],[1.   ],[0.   ],[0.375],[1.   ]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled)
elastic_model = ElasticNetSubspaceClustering(n_clusters=4,
algorithm='lasso_lars',gamma=1)
elastic_model.fit(X_scaled)
y_predit = elastic_model.labels_

#y_predit,_ =RunKMeanCluster(X_scaled, 2)
player_dataset['GroupID'] = y_predit

print(player_dataset[['userID', 'GroupID']])


In [ ]:
import shutil
def move_files_with_underscore(path, target_folder):
    # Ensure the target folder exists
    os.makedirs(target_folder, exist_ok=True)

    for file in os.listdir(path):
        if "_" in file:
            source_path = os.path.join(path, file)
            destination_path = os.path.join(target_folder, file)
            shutil.move(source_path, destination_path)
            print(f"Moved '{file}' to '{target_folder}'")


 { "Features": { "WalkPath":true, "GameplayDur":false, "NumCoinsCollected":true, "NumBombHit":false, "NumObstaclesHit":false, "NumTreasureCollected":false}, "Pose_Metric": "Euclidean", "Pose_Algo":  "ElasticNet","NumCluster" : "2" }